In [2]:
from sklift.datasets import fetch_hillstrom
bunch = fetch_hillstrom(target_col='visit')
print(bunch.data.shape)

(64000, 8)


In [3]:
X = bunch.data
y = bunch.target
treat = bunch.treatment

X.head()

,recency,history_segment,history,mens,womens,zip_code,newbie,channel
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web


In [4]:
treat.value_counts()

segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

In [5]:
df = X.copy()
df['treatment'] = treat
df['visit'] = y

df.groupby('treatment')[['recency', 'history']].mean()

,recency,history
treatment,,
Mens E-Mail,5.773642,242.835931
No E-Mail,5.749695,240.882653
Womens E-Mail,5.767850,242.536633


In [6]:
import numpy as np

def smd(col, g1='Womens E-Mail', g2='No E-Mail'):
    a = df[df['treatment'] == g1][col]
    b = df[df['treatment'] == g2][col]
    pooled_sd = np.sqrt((a.std()**2 + b.std()**2) / 2)
    return (a.mean() - b.mean()) / pooled_sd

for c in ['recency', 'history', 'mens', 'womens', 'newbie']:
    print(f"{c:10s} SMD = {smd(c):+.4f}")

recency    SMD = +0.0052
history    SMD = +0.0065
mens       SMD = -0.0086
womens     SMD = +0.0049
newbie     SMD = +0.0026


In [7]:
df.groupby('treatment')['visit'].agg(['count', 'mean'])

,count,mean
treatment,,
Mens E-Mail,21307,0.182757
No E-Mail,21306,0.106167
Womens E-Mail,21387,0.151400


In [8]:
df[['mens', 'womens']].mean()

mens      0.551031
womens    0.549719
dtype: float64

In [9]:
df.groupby(['treatment', 'mens'])['visit'].mean().unstack()

mens,0,1
treatment,,
Mens E-Mail,0.166388,0.196098
No E-Mail,0.095808,0.114533
Womens E-Mail,0.169794,0.136286


In [10]:
import numpy as np

df['买过'] = np.where((df.mens == 1) & (df.womens == 1), '都买',
             np.where(df.mens == 1, '只买男装', '只买女装'))

df.groupby(['treatment', '买过'])['visit'].agg(['count', 'mean']).unstack()

count                  mean                    
买过             只买女装  只买男装    都买      只买女装      只买男装        都买
treatment                                                    
Mens E-Mail    9568  9558  2181  0.166388  0.168968  0.314993
No E-Mail      9519  9638  2149  0.095808  0.099813  0.180549
Womens E-Mail  9647  9622  2118  0.169794  0.110892  0.251653